In [1]:
import folium
import pandas as pd

from folium.features import DivIcon

/Users/hmc123/Documents/scheduling/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# pd.set_option('display.max.columns', 50)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 1000)

In [3]:
df = pd.read_excel('얘선데이터_인천항.xlsx')
berth_df = pd.read_csv("./data/정계지 위치.csv")
df_선석 = pd.read_csv("./data/선석 코드.csv")

In [41]:
"NPPS" in list(df_선석.선석코드.unique())

False

In [4]:
lat, lon = berth_df["위도"].mean(), berth_df["경도"].mean()
map_obj = folium.Map(location=[lat, lon], zoom_start=13)

# CircleMarker 내부에 텍스트 표시
for _, _val in berth_df.iterrows():
    _lat, _lon = _val["위도"], _val["경도"]
    _name = _val["정계지"][:2]
    
    folium.CircleMarker(
        location=[_lat, _lon],
        radius=30,
        color="",
        fill=True,
        fill_color="#ff7800",
        fill_opacity=0.3
    ).add_to(map_obj)
    
    folium.map.Marker(
        [_lat, _lon],
        icon=DivIcon(
            icon_size=(60, 20),  # 아이콘 크기 조정
            icon_anchor=(30, 10),  # 앵커 위치 조정
            html=f'<div style="font-size: 8pt; color : black; text-align: center; font-weight: bold;">{_name}</div>',
        )
    ).add_to(map_obj)

In [26]:
map_obj

In [5]:
# 데이터프레임의 정보 출력
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336 entries, 0 to 335
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   기준예선            336 non-null    object 
 1   배정예선            336 non-null    object 
 2   선박명             336 non-null    object 
 3   From            217 non-null    object 
 4   작업 시작지          336 non-null    object 
 5   작업 종료지          336 non-null    object 
 6   To              190 non-null    object 
 7   실제 스케줄 시작 시각    336 non-null    object 
 8   실제 스케줄 종료 시각    336 non-null    object 
 9   작업까지 이동 거리(km)  336 non-null    float64
 10  작업까지 이동 시간(분)   336 non-null    float64
 11  작업중 이동 거리(km)   336 non-null    float64
 12  작업중 이동 시간(분)    336 non-null    float64
 13  도선사             252 non-null    object 
 14  톤수              336 non-null    int64  
 15  선종              336 non-null    object 
 16  최종 스케줄 시각       336 non-null    object 
 17  최초 스케줄 시각       336 non-null    obj

In [36]:
import random

# NaN이 아닌 From 값들 중 랜덤하게 선택하여 NaN 값 대체
df['From'] = df['From'].apply(lambda x: random.choice(df['From'].dropna().unique()) if pd.isna(x) else x)
df['To'] = df['To'].apply(lambda x: random.choice(df['To'].dropna().unique()) if pd.isna(x) else x)

In [44]:
df_선석

,선석코드,위도,경도
0,KICT,37.561339,126.597451
1,KIPT,37.558609,126.605923
2,KIST,37.560915,126.612583
3,CP,37.440009,126.587046
4,ICT1,37.438315,126.599400
...,...,...,...
106,80,37.475815,126.612962
107,81,37.474601,126.613466
108,82,37.472364,126.614527
109,84,37.476689,126.611032


In [45]:
import random

# df_선석의 선석코드와 df의 작업 시작지, 작업 종료지를 join
df = df.merge(df_선석, left_on='작업 시작지', right_on='선석코드', how='left')
df = df.merge(df_선석, left_on='작업 종료지', right_on='선석코드', how='left', suffixes=('_시작', '_종료'))

# 작업 시작지, 작업 종료지에 해당하는 선석코드가 없으면 랜덤으로 대체
df['작업 시작지'] = df['작업 시작지'].apply(
    lambda x: random.choice(df_선석['선석코드'].dropna().unique()) if pd.isna(x) else x
)
df['작업 종료지'] = df['작업 종료지'].apply(
    lambda x: random.choice(df_선석['선석코드'].dropna().unique()) if pd.isna(x) else x
)


In [46]:
df

,기준예선,배정예선,선박명,From,작업 시작지,작업 종료지,To,실제 스케줄 시작 시각,실제 스케줄 종료 시각,작업까지 이동 거리(km),작업까지 이동 시간(분),작업중 이동 거리(km),작업중 이동 시간(분),도선사,톤수,선종,최종 스케줄 시각,최초 스케줄 시각,최초 작업시작지,최초 작업종료지,최초 예선,최초 도선사,AIS 파일명 target,AIS 파일명 tug,선석코드_시작,위도_시작,경도_시작,선석코드_종료,위도_종료,경도_종료
0,I,"성,I",REVERENCE,송도정계지,NPPS,SNCT2,송도정계지,2024-06-08T07:32:18.000Z,2024-06-08T08:15:18.000Z,8.571,30.000000,4.909,38.000000,KU,9990,풀컨테이너선,2024-06-08T07:00:00.000Z,2024-06-08T07:00:00.000Z,NPPS,SNCT2,성I,KU,쥬피터1호_REVERENCE_2024-06-08_target_1_aisData.csv,쥬피터1호_REVERENCE_2024-06-08_tug_1_aisData.csv,NaN,NaN,NaN,SNCT2,37.341496,126.637137
1,M,"F1,M",SEAWAYS MOMENT,연안부두정계지,SKI 2,DONG,연안부두정계지,2024-06-08T07:10:18.000Z,2024-06-08T07:20:18.000Z,0.626,2.000000,1.859,5.000000,JJ,29293,케미칼 운반선,2024-06-08T07:00:00.000Z,2024-06-08T07:00:00.000Z,SKI 2,DONG,F1M,JJ,뉴마스호_SEAWAYS MOMENT_2024-06-08_target_2_aisData.csv,뉴마스호_SEAWAYS MOMENT_2024-06-08_tug_2_aisData.csv,NaN,NaN,NaN,NaN,NaN,NaN
2,T,"T,S",YURIY TARAPUROV,연안부두정계지,PAL,N53,송도정계지,2024-06-08T10:51:18.000Z,2024-06-08T11:01:19.000Z,5.673,25.983333,2.561,35.016667,CY,7949,일반화물선,2024-06-08T09:20:00.000Z,2024-06-08T09:00:00.000Z,PAL,N53,TS,CY,대창호_YURIY TARAPUROV_2024-06-08_target_3_aisData.csv,대창호_YURIY TARAPUROV_2024-06-08_tug_3_aisData.csv,NaN,NaN,NaN,N53,37.493756,126.628681
3,M,M,BIRYONG,연안부두정계지,PAL,NIPT2,내항정계지,2024-06-08T10:26:18.000Z,2024-06-08T11:01:18.000Z,5.235,47.000000,3.371,30.000000,KI,14614,국제카페리,2024-06-08T09:50:00.000Z,2024-06-08T09:30:00.000Z,PAL,NIPT2,M,KI,뉴마스호_BIRYONG_2024-06-08_target_4_aisData.csv,뉴마스호_BIRYONG_2024-06-08_tug_4_aisData.csv,NaN,NaN,NaN,NIPT2,37.423926,126.601297
4,M,M,XIN XIANG XUE LAN,내항정계지,PAL,NIPT5,연안부두정계지,2024-06-08T11:20:18.000Z,2024-06-08T11:59:18.000Z,4.024,26.983333,4.567,34.000000,JJ,32729,국제카페리,2024-06-08T10:50:00.000Z,2024-06-08T10:30:00.000Z,PAL,NIPT5,M,NaN,뉴마스호_XIN XIANG XUE LAN_2024-06-08_target_5_aisData.csv,뉴마스호_XIN XIANG XUE LAN_2024-06-08_tug_5_aisData.csv,NaN,NaN,NaN,NIPT5,37.420416,126.599211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,I,I,DONGJIN CONTINENTAL,송도정계지,HJIT3,NPPS,연안부두정계지,2024-06-29T22:25:21.000Z,2024-06-29T22:36:21.000Z,8.930,989.000000,0.863,10.000000,NaN,9946,풀컨테이너선,2024-06-29T22:00:00.000Z,2024-06-29T07:00:00.000Z,NPPS,HJIT3,H,NaN,쥬피터1호_DONGJIN CONTINENTAL_2024-06-29_target_332_aisData.csv,쥬피터1호_DONGJIN CONTINENTAL_2024-06-29_tug_332_aisData.csv,HJIT3,37.341566,126.631708,NaN,NaN,NaN
332,미,"E,미",WOO CHOON,연안부두정계지,GSD,PAL,연안부두정계지,2024-06-30T13:44:21.000Z,2024-06-30T14:10:21.000Z,3.232,13.000000,2.817,21.000000,NaN,3969,석유제품 운반선,2024-06-30T14:00:00.000Z,2024-06-30T11:30:00.000Z,GSD,PAL,E미,NaN,뉴월미호_WOO CHOON_2024-06-30_target_333_aisData.csv,뉴월미호_WOO CHOON_2024-06-30_tug_333_aisData.csv,GSD,37.478154,126.594861,NaN,NaN,NaN
333,N,N,DB SUNNY,연안부두정계지,W4,HIT,연안부두정계지,2024-06-30T19:38:22.000Z,2024-06-30T20:11:22.000Z,6.449,339.000000,3.369,28.000000,NaN,1599,케미칼 운반선,2024-06-30T19:00:00.000Z,2024-06-30T20:00:00.000Z,W4,HIT,N,NaN,뉴엔젤호_DB SUNNY_2024-06-30_target_334_aisData.csv,뉴엔젤호_DB SUNNY_2024-06-30_tug_334_aisData.csv,NaN,NaN,NaN,HIT,37.445362,126.615208
334,M,"M,S",OCEAN GAS,연안부두정계지,PAL,SKI 1,내항정계지,2024-06-30T18:53:22.000Z,2024-06-30T19:46:22.000Z,6.748,39.983333,5.252,48.000000,NaN,3678,LPG 운반선,2024-06-30T18:30:00.000Z,2024-06-30T18:30:00.000Z,PAL,SKI 1,MW,NaN,뉴마스호_OCEAN GAS_2024-06-30_target_335_aisData.csv,뉴마스호_OCEAN GAS_2024-06-30_tug_335_aisData.csv,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
for _idx, _row in df.iterrows():
    print(_row)
    break

기준예선                                                            I
배정예선                                                          성,I
선박명                                                     REVERENCE
From                                                          NaN
작업 시작지                                                       NPPS
                                       ...                       
최초 작업종료지                                                    SNCT2
최초 예선                                                          성I
최초 도선사                                                         KU
AIS 파일명 target    쥬피터1호_REVERENCE_2024-06-08_target_1_aisData.csv
AIS 파일명 tug          쥬피터1호_REVERENCE_2024-06-08_tug_1_aisData.csv
Name: 0, Length: 24, dtype: object


In [43]:
df.To.unique()

array(['송도정계지', '연안부두정계지', '내항정계지', '북항임시정계지'], dtype=object)